In [ ]:
# Classical Cross-Domain Experiments

# This notebook evaluates how the training domain affects CWE classification on the same fixed Russian test set.
# Three training conditions are compared: Russian, non-Russian, and their combination.

In [1]:
import pandas as pd

from src.config import (
    TEXT_COL,
    LABEL_COL,
    ID_COL,
    CWE_LABELS,
    RESULTS_MAIN_DIR,
    RESULTS_PER_CLASS_DIR,
    RESULTS_PREDICTIONS_DIR,
)

from src.data import (
    load_preprocessed_data,
    assert_no_group_overlap,
)

from src.models import (
    fit_predict_classical,
)

from src.evaluation import (
    evaluate_predictions,
    domain_generalisation_gap,
    combined_training_gain,
)


ru_df, nonru_df = (
    load_preprocessed_data()
)


ru_dev = (
    ru_df[
        ru_df["split"].isin(
            ["train", "val"]
        )
    ]
    .copy()
)

ru_test = (
    ru_df[
        ru_df["split"] == "test"
    ]
    .copy()
)

nonru_dev = (
    nonru_df[
        nonru_df["split"].isin(
            ["train", "val"]
        )
    ]
    .copy()
)

combined_dev = pd.concat(
    [
        ru_dev,
        nonru_dev
    ],
    ignore_index=True
)


print("RU:", len(ru_dev))
print("Non-RU:", len(nonru_dev))
print("Combined:", len(combined_dev))
print("RU test:", len(ru_test))

RU: 12280
Non-RU: 41628
Combined: 53908
RU test: 2171


In [ ]:
## Verify Test-Set Isolation

# The non-Russian and combined development datasets are checked against the fixed Russian test set
# to ensure that no leakage group is shared with the evaluation data.

In [2]:
assert_no_group_overlap(
    nonru_dev,
    ru_test
)

assert_no_group_overlap(
    combined_dev,
    ru_test
)

print(
    "Fixed RU test remains isolated."
)

Fixed RU test remains isolated.


In [ ]:
## Run the Cross-Domain Experiments

# The selected Logistic Regression and Linear SVM configurations are trained under the Russian,
# non-Russian, and combined conditions. Every model is evaluated against the identical Russian test set.

In [3]:
conditions = {
    "RU":
        ru_dev,

    "Non-RU":
        nonru_dev,

    "Combined":
        combined_dev,
}


models = {
    "Logistic Regression":
        "word",

    "Linear SVM":
        "char",
}


results = []
per_class_results = {}
predictions = {}


for model_name, representation in (
    models.items()
):

    for condition, train_df in (
        conditions.items()
    ):

        pred, n_features, runtime = (
            fit_predict_classical(
                train_df,
                ru_test,
                model_name,
                representation,
                TEXT_COL,
                LABEL_COL,
            )
        )

        metrics, per_class = (
            evaluate_predictions(
                ru_test[LABEL_COL],
                pred
            )
        )

        results.append({
            "model":
                model_name,

            "representation":
                representation,

            "training_condition":
                condition,

            "train_records":
                len(train_df),

            "test_records":
                len(ru_test),

            **metrics,

            "n_features":
                n_features,

            "runtime_seconds":
                runtime,
        })


        key = (
            f"{model_name}_{condition}"
        )

        per_class_results[
            key
        ] = per_class

        predictions[
            key
        ] = pd.DataFrame({
            ID_COL:
                ru_test[ID_COL].values,

            "true_cwe":
                ru_test[LABEL_COL].values,

            "prediction":
                pred,

            "model":
                model_name,

            "training_condition":
                condition,
        })


results_df = pd.DataFrame(
    results
)

results_df

,model,representation,training_condition,train_records,test_records,macro_f1,weighted_f1,accuracy,n_features,runtime_seconds
0,Logistic Regression,word,RU,12280,2171,0.866775,0.891821,0.889452,14943,4.092214
1,Logistic Regression,word,Non-RU,41628,2171,0.826679,0.859282,0.861815,46257,20.771775
2,Logistic Regression,word,Combined,53908,2171,0.877897,0.900085,0.900507,56776,23.305718
3,Linear SVM,char,RU,12280,2171,0.884289,0.903213,0.902810,78459,14.487624
4,Linear SVM,char,Non-RU,41628,2171,0.835386,0.866799,0.867803,136548,51.320929
5,Linear SVM,char,Combined,53908,2171,0.884094,0.900458,0.900967,173354,65.575849


In [ ]:
## Quantify Domain Effects

# Macro-F1 results are converted into the two central comparison measures:
# the cross-domain generalisation gap and the gain or loss produced by combined-domain training.

In [4]:
comparison = []


for model_name in models:

    current = (
        results_df[
            results_df["model"]
            == model_name
        ]
        .set_index(
            "training_condition"
        )
    )


    ru_macro = (
        current.loc[
            "RU",
            "macro_f1"
        ]
    )

    nonru_macro = (
        current.loc[
            "Non-RU",
            "macro_f1"
        ]
    )

    combined_macro = (
        current.loc[
            "Combined",
            "macro_f1"
        ]
    )


    comparison.append({

        "model":
            model_name,

        "RU_to_RU_macro_f1":
            ru_macro,

        "NonRU_to_RU_macro_f1":
            nonru_macro,

        "Combined_to_RU_macro_f1":
            combined_macro,

        "domain_generalisation_gap":
            domain_generalisation_gap(
                ru_macro,
                nonru_macro
            ),

        "combined_training_gain":
            combined_training_gain(
                ru_macro,
                combined_macro
            ),
    })


central_comparison = pd.DataFrame(
    comparison
)

central_comparison

,model,RU_to_RU_macro_f1,NonRU_to_RU_macro_f1,Combined_to_RU_macro_f1,domain_generalisation_gap,combined_training_gain
0,Logistic Regression,0.866775,0.826679,0.877897,0.040096,0.011122
1,Linear SVM,0.884289,0.835386,0.884094,0.048903,-0.000195


In [ ]:
## Save Cross-Domain Results

# Aggregate metrics, per-CWE performance, and record-level predictions are exported
# for the main dissertation results and subsequent error analysis.

In [6]:
results_df.to_csv(
    RESULTS_MAIN_DIR
    / "classical_cross_domain_all_results.csv",
    index=False
)

central_comparison.to_csv(
    RESULTS_MAIN_DIR
    / "classical_cross_domain_central_comparison.csv",
    index=False
)


for key, dataframe in predictions.items():

    safe_name = (
        key
        .replace(" ", "_")
    )

    dataframe.to_csv(
        RESULTS_PREDICTIONS_DIR
        / f"predictions_{safe_name}.csv",
        index=False
    )


for key, dataframe in (
    per_class_results.items()
):

    safe_name = (
        key
        .replace(" ", "_")
    )

    dataframe.to_csv(
        RESULTS_PER_CLASS_DIR
        / f"per_class_{safe_name}.csv",
        index=False
    )


print("Classical domain results saved.")

Classical domain results saved.
